In [1]:
import torch
import optuna, json
import numpy as np
from fem_utilities.fem_fenics import *
from scipy.sparse.linalg import spsolve
from neural_net.training import *
from neural_net.networks import FullyConnected
from sklearn import preprocessing
import pickle

In [2]:
torch.manual_seed(25)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

### 1. Extração da matriz de rigidez K e vetor fonte F

In [3]:
# pontos_laser = [(0.05, 0.05)]
# A_pontos     = 1.3e6

# pontos_laser = [(0.045, 0.045), (0.055, 0.055)]
# A_pontos     = 1.3e6/2

pontos_laser = [(0.045, 0.045), (0.055, 0.045), (0.045, 0.055), (0.055, 0.055)]
A_pontos     = 1.3e6/4

In [4]:
bioheat1 = BioHeatSimulation('mesh/malha.xdmf')

K, F = bioheat1.extract_system_matrices(pontos_laser, A_pontos)
F    = F.reshape(-1, 1)

print("Shape da matriz A:", K.shape)
print("Shape do vetor b:", F.shape)

K_torch = torch.sparse_csr_tensor(crow_indices=K.indptr, col_indices=K.indices, values=K.data, size=K.shape).to(device)
F_torch = torch.tensor(F, dtype=torch.double).to(device)

Shape da matriz A: (89, 89)
Shape do vetor b: (89, 1)


/tmp/ipykernel_23936/56870158.py:9: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  K_torch = torch.sparse_csr_tensor(crow_indices=K.indptr, col_indices=K.indices, values=K.data, size=K.shape).to(device)
/tmp/ipykernel_23936/56870158.py:9: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  K_torch = torch.sparse_csr_tensor(crow_indices=K.indptr, col_indices=K.indices, values=K.data, size=K.shape).to(device)


In [5]:
# def scipy_to_torch_sparse(A):
#     A_coo   = A.tocoo()
#     values  = A_coo.data
#     indices = np.vstack((A_coo.row, A_coo.col))
    
#     i     = torch.LongTensor(indices)
#     v     = torch.DoubleTensor(values) # Usando Double já que você usou .double() na rede
#     shape = A_coo.shape
    
#     return torch.sparse_coo_tensor(i, v, torch.Size(shape)).to(device)

# K_torch = scipy_to_torch_sparse(K)
# F_torch = torch.tensor(F, dtype=torch.double).to(device)

### 2. Preparando dados para treinamento

In [6]:
data_in = bioheat1.extract_nodes()

scaler         = preprocessing.MinMaxScaler()
data_in_scaled = scaler.fit_transform(data_in)
pickle.dump(scaler, open('scaler.pkl', 'wb'))

data_in_torch = torch.as_tensor(data_in_scaled, dtype=torch.double).to(device)
data_in_torch.shape

torch.Size([89, 2])

### 3. Buscando os melhores hiperparâmetros para a rede com `Optuna`

In [7]:
# def objective(trial):
#     # Suggest values of the hyperparameters using a trial object.
#     depth    = trial.suggest_int('depth', 3, 8)
#     n_hidden = trial.suggest_int('n_hidden', 32, 256)
#     init     = trial.suggest_float('init', 0.1, 2)
#     lr       = trial.suggest_float('lr', 1.0E-5, 1.0E-2, log=True)

#     # Creation of network
#     net_model        = FullyConnected(n_hidden=n_hidden, depth=depth, init=init).double().to(device)
#     fem_loss_trainer = TrainFemLoss(net_model, K_torch, F_torch)
#     train_loss       = fem_loss_trainer.train(data_in_torch, lr, epochs=100)

#     return train_loss

# # Create a study object and optimize the objective function to find the best hyperparameters.
# study = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner())
# study.optimize(objective, n_trials=500)

In [8]:
# print(study.best_params['depth'], study.best_params['n_hidden'], study.best_params['init'], study.best_params['lr'])

In [9]:
# # Após o study.optimize
# melhores_params = study.best_params

# # Adicionar informações extras úteis
# dados_modelo = {
#     'params': melhores_params,
#     'best_value': study.best_value, # A menor perda encontrada
# }

# # Salvar em arquivo
# with open('hyper_params.json', 'w') as f:
#     json.dump(dados_modelo, f, indent=4)

# print('Parâmetros salvos com sucesso!')

### 4. Treinamento com os Hiperparâmetros determinados

In [10]:
with open('hyper_params.json', 'r') as f:
    config = json.load(f)

params = config['params']

In [11]:
model = FullyConnected(params['n_hidden'], params['depth'], params['init']).double().to(device)

In [12]:
# Treino usando FEM-NN
epoch            = 10000
fem_loss_trainer = TrainFemLoss(model, K_torch, F_torch)
train_loss       = fem_loss_trainer.train(data_in_torch, params['lr'], epochs=epoch)

Epoch: 0, Train Loss: 1.61790E+02
Epoch: 1000, Train Loss: 2.63686E+00
Epoch: 2000, Train Loss: 3.11262E+00
Epoch: 3000, Train Loss: 1.27865E+00
Epoch: 4000, Train Loss: 2.28114E-01
Epoch: 5000, Train Loss: 6.79927E-02
Epoch: 6000, Train Loss: 1.91547E-02
Epoch: 7000, Train Loss: 4.39865E-03
Epoch: 8000, Train Loss: 1.14209E-03
Epoch: 9000, Train Loss: 2.81784E-04


In [13]:
# Salvando o modelo
torch.save(model.state_dict(), 'models/model.pt')

### 5. Utilizando o modelo salvo para fazer predições

In [14]:
# Preparando os dados para teste do modelo

bioheat2 = BioHeatSimulation('mesh/malha_fina.xdmf')

data_in_test = bioheat2.extract_nodes()

scalar              = pickle.load(open('scaler.pkl', 'rb'))
data_in_test_scaled = scalar.transform(data_in_test)
input_data          = torch.as_tensor(data_in_test_scaled, dtype=torch.double).to(device)

input_data.shape

torch.Size([1093, 2])

In [15]:
# Definindo a arquitetura do modelo e avaliando na nova malha

modelo = FullyConnected(params['n_hidden'], params['depth'], params['init']).double().to(device)
modelo.load_state_dict(torch.load('models/model.pt'))

modelo.eval()
with torch.inference_mode():
    T_femnn = modelo(input_data).detach().cpu().numpy()

T_femnn.shape

(1093, 1)

In [16]:
# Visualizando a solução com Pyvista
bioheat2.plot_solution(T_femnn)

Widget(value='<iframe src="http://localhost:42863/index.html?ui=P_0x740ecde6d6a0_0&reconnect=auto" class="pyvi…

### 6. Solução FEM com FEniCS

In [17]:
T_fen = bioheat2.run_simulation(pontos_laser, A_pontos)
bioheat2.plot_solution(T_fen)


Simulação Concluída
Temperatura Máxima: 46.84 °C


Widget(value='<iframe src="http://localhost:42863/index.html?ui=P_0x740ec1e52fd0_1&reconnect=auto" class="pyvi…